In [8]:
# Core libraries
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
import os

In [13]:
# The companies we're tracking, grouped by sector
tickers = {
    'glp1_makers':      ['LLY', 'NVO'],
    'diabetes_devices': ['DXCM', 'PODD'],
    'food_beverage':    ['KHC', 'NSRGY', 'PEP'],
    'fast_food':        ['MCD', 'YUM'],
    'fitness':          ['PLNT'],
    'surgical':         ['ISRG']
}

# FDA approval dates — these are our analytical anchors
events = {
    'wegovy_approved':   '2021-06-04',  # Wegovy for obesity
    'mounjaro_approved': '2022-05-13',  # Mounjaro for diabetes  
    'zepbound_approved': '2023-11-08'   # Zepbound for obesity
}

# Flatten tickers into a single list for data pulling
all_tickers = [t for group in tickers.values() for t in group]

print("Tickers we're tracking:", all_tickers)
print("Number of companies:", len(all_tickers))
print("Event dates:", events)

Tickers we're tracking: ['LLY', 'NVO', 'DXCM', 'PODD', 'KHC', 'NSRGY', 'PEP', 'MCD', 'YUM', 'PLNT', 'ISRG']
Number of companies: 11
Event dates: {'wegovy_approved': '2021-06-04', 'mounjaro_approved': '2022-05-13', 'zepbound_approved': '2023-11-08'}


In [17]:
# Pull historical price data for all tickers
# Starting from 2019 to give us 2 years of baseline before the first FDA event
print("Pulling stock data... this may take a moment")

price_data = yf.download(
    tickers=all_tickers,
    start='2019-01-01',
    end=datetime.today().strftime('%Y-%m-%d'),
    auto_adjust=True
)

# We only need the closing price
close_prices = price_data['Close']

print(f"\nData pulled successfully!")
print(f"Date range: {close_prices.index[0].date()} to {close_prices.index[-1].date()}")
print(f"Shape: {close_prices.shape[0]} trading days x {close_prices.shape[1]} companies")
print(f"\nFirst few rows:")
close_prices.head()

[*********************100%***********************]  11 of 11 completed

Pulling stock data... this may take a moment

Data pulled successfully!
Date range: 2019-01-02 to 2026-07-01
Shape: 1884 trading days x 11 companies

First few rows:


Ticker,DXCM,ISRG,KHC,LLY,MCD,NSRGY,NVO,PEP,PLNT,PODD,YUM
Date,,,,,,,,,,,
2019-01-02,28.795000,155.343338,29.806629,104.162521,147.795364,65.293228,19.790524,86.693657,53.580002,73.430000,79.630615
2019-01-03,28.065001,150.080002,29.786001,100.925621,146.821594,66.954659,19.951250,85.884483,52.799999,72.440002,77.627632
2019-01-04,29.059999,157.226669,30.597525,103.963058,149.658951,68.078590,20.023157,87.645653,54.919998,77.150002,79.648010
2019-01-07,32.487499,159.479996,31.175234,104.525261,151.287521,67.361893,20.150049,86.891991,55.840000,74.209999,79.560913
2019-01-08,32.880001,160.996674,31.202742,105.486351,151.606522,67.573647,20.446119,87.724991,57.619999,74.150002,79.404175


In [20]:
# Check data quality before saving
print("=== DATA QUALITY CHECK ===\n")

# Check for missing values
missing = close_prices.isnull().sum()
print("Missing values per ticker:")
print(missing)

print(f"\nTotal missing values: {missing.sum()}")
print(f"\nPercentage complete per ticker:")
completeness = ((len(close_prices) - missing) / len(close_prices) * 100).round(2)
print(completeness)

=== DATA QUALITY CHECK ===

Missing values per ticker:
Ticker
DXCM     0
ISRG     0
KHC      0
LLY      0
MCD      0
NSRGY    0
NVO      0
PEP      0
PLNT     0
PODD     0
YUM      0
dtype: int64

Total missing values: 0

Percentage complete per ticker:
Ticker
DXCM     100.0
ISRG     100.0
KHC      100.0
LLY      100.0
MCD      100.0
NSRGY    100.0
NVO      100.0
PEP      100.0
PLNT     100.0
PODD     100.0
YUM      100.0
dtype: float64


In [28]:
import os

# Project root
project_root = '/Users/nicoleotero/Desktop/GLP1 research project/glp1-market-impact-analysis'
save_path = os.path.join(project_root, 'data', 'raw', 'close_prices_raw.csv')

# Save it
close_prices.to_csv(save_path)

print(f"Saved successfully to: {save_path}")
print(f"File contains {close_prices.shape[0]} rows and {close_prices.shape[1]} columns")

Saved successfully to: /Users/nicoleotero/Desktop/GLP1 research project/glp1-market-impact-analysis/data/raw/close_prices_raw.csv
File contains 1884 rows and 11 columns


In [29]:
# Save event dates as a CSV too
events_df = pd.DataFrame([
    {'event': k, 'date': v} for k, v in events.items()
])

events_save_path = os.path.join(project_root, 'data', 'raw', 'fda_event_dates.csv')
events_df.to_csv(events_save_path, index=False)

print("FDA event dates saved successfully!")
print(events_df)

FDA event dates saved successfully!
               event        date
0    wegovy_approved  2021-06-04
1  mounjaro_approved  2022-05-13
2  zepbound_approved  2023-11-08


In [30]:
# Pull SPY as market benchmark
import yfinance as yf
import pandas as pd
import os

project_root = '/Users/nicoleotero/Desktop/GLP1 research project/glp1-market-impact-analysis'

spy = yf.download('SPY', start='2019-01-01', 
                  end=pd.Timestamp.today().strftime('%Y-%m-%d'),
                  auto_adjust=True)['Close']

spy.columns = ['SPY']

# Save to raw folder
spy_path = os.path.join(project_root, 'data', 'raw', 'spy_benchmark.csv')
spy.to_csv(spy_path)

print("SPY benchmark data saved successfully")
print(f"Date range: {spy.index[0].date()} to {spy.index[-1].date()}")
spy.head()

[*********************100%***********************]  1 of 1 completed

SPY benchmark data saved successfully
Date range: 2019-01-02 to 2026-07-01


,SPY
Date,
2019-01-02,223.805969
2019-01-03,218.465347
2019-01-04,225.782974
2019-01-07,227.563232
2019-01-08,229.701187
